In [17]:
from typing import Final, Literal
import sys
sys.path.insert(1, '/home/spuchin/GitHub/the-hand-of-midas')

from pandas import DataFrame, read_csv
import plotly.graph_objects as go

In [18]:
ohlc: DataFrame = read_csv("/home/spuchin/GitHub/the-hand-of-midas/notebooks/DS-2/global-trend-lines.csv")
ohlc.tail(1)

,open,high,low,close,datetime,open_macro,high_macro,low_macro,close_macro,open_macro_to_high_global_trend_ratio,open_macro_to_low_global_trend_ratio,high_macro_to_high_global_trend_ratio,high_macro_to_low_global_trend_ratio,low_macro_to_high_global_trend_ratio,low_macro_to_low_global_trend_ratio,close_macro_to_high_global_trend_ratio,close_macro_to_low_global_trend_ratio
17147,107424.96,107684.13,107309.15,107684.13,2025-06-16 16:00:00,106080.455249,106582.982636,105581.064839,106161.916044,0.018818,0.029947,0.023644,0.034826,0.014022,0.025098,0.0196,0.030738


In [19]:
candle_columns: Final[list[str]] = ["open", "high", "low", "close"]

In [20]:
def _is_exclude_column(column: str, columns_to_exclude: list[str]) -> bool:
    return column == "datetime" or column in columns_to_exclude or column.startswith("is") or column.endswith("streak")

value_columns: list[str] = []
for column in list(ohlc.columns):
    if not _is_exclude_column(column=column, columns_to_exclude=candle_columns):
        value_columns.append(column)

### DS-8: [feature] Add streak for boolean features.

In [21]:
# https://joshdevlin.com/blog/calculate-streaks-in-pandas/
def _streak_by(dataframe: DataFrame, boolean_column: str) -> DataFrame:
    dataframe["__start_of_streak"] = dataframe[boolean_column].ne(dataframe[boolean_column].shift(1))
    dataframe["__streak_id"] = dataframe["__start_of_streak"].cumsum()
    
    dataframe[f"{boolean_column[3:]}_streak"] = dataframe.groupby("__streak_id").cumcount() + 1

    dataframe.drop(labels=["__start_of_streak", "__streak_id"], axis=1, inplace=True)
    return dataframe

In [22]:
def _is_greater(first_number: float, second_number: float) -> int:
    """Function to create boolean columns based on comparison between previous and current values."""
    return 1 if first_number > second_number else 0

### DS-9: [feature] Add difference from first value.

In [23]:
def _difference_from_first(dataframe: DataFrame, boolean_column: str, value_column: str) -> DataFrame:
    """Dynamic changing from start of the streak, %."""

    dataframe["__start_of_streak"] = dataframe[boolean_column].ne(dataframe[boolean_column].shift(1))
    dataframe["__streak_id"] = dataframe["__start_of_streak"].cumsum()
    
    first_value_in_streak = dataframe.groupby("__streak_id")[value_column].first().reset_index().rename(columns={value_column: "__first_value_in_streak"})
    dataframe = dataframe.merge(first_value_in_streak, how="left", on="__streak_id")

    # because of `previous_` prefix we take 9-first symbols and 3-first symbols for `boolean_column` because of `is_` prefix 
    boolean_column_to_represent: str = boolean_column[3:]
    value_column_to_represent: str = value_column[9:] if value_column.startswith('previous_') else value_column
    
    dataframe[f"{boolean_column_to_represent}_streak_{value_column_to_represent}_difference_from_first_ratio"] = dataframe[value_column] / dataframe["__first_value_in_streak"] - 1

    dataframe.drop(labels=["__start_of_streak", "__streak_id", "__first_value_in_streak"], axis=1, inplace=True)
    return dataframe

### DS-3

In [24]:
# macro candle streak
ohlc["is_macro_close_greater_than_macro_open"] = ohlc.apply(
    lambda row: _is_greater(
        first_number=row["close_macro"], 
        second_number=row["open_macro"]
    ),
    axis=1
)
ohlc = _streak_by(dataframe=ohlc, boolean_column="is_macro_close_greater_than_macro_open")

# diff base
for value_column in value_columns:
    ohlc = _difference_from_first(dataframe=ohlc, boolean_column="is_macro_close_greater_than_macro_open", value_column=value_column)

# diff for noise-candles
for candle_column in candle_columns:
    ohlc[f"previous_{candle_column}"] = ohlc[candle_column].shift(1)
    ohlc = _difference_from_first(dataframe=ohlc, boolean_column="is_macro_close_greater_than_macro_open", value_column=f"previous_{candle_column}")
    ohlc.drop(labels=[f"previous_{candle_column}"], axis=1, inplace=True)

### DS-2

In [25]:
columns: list[str] = [
    "open_macro_to_high_global_trend_ratio", 
    "open_macro_to_low_global_trend_ratio", 
    "high_macro_to_high_global_trend_ratio", 
    "high_macro_to_low_global_trend_ratio",

    "low_macro_to_high_global_trend_ratio",
    "low_macro_to_low_global_trend_ratio",
    "close_macro_to_high_global_trend_ratio",
    "close_macro_to_low_global_trend_ratio"
]

In [26]:
# global trend streak
for column in columns:
    ohlc[f"is_{column}_greater_than_zero"] = ohlc[column].apply(
        lambda value: _is_greater(
            first_number=value, 
            second_number=0
        )
    )
    ohlc = _streak_by(dataframe=ohlc, boolean_column=f"is_{column}_greater_than_zero")

    # diff base
    for value_column in value_columns:
        ohlc = _difference_from_first(dataframe=ohlc, boolean_column=f"is_{column}_greater_than_zero", value_column=value_column)

    # diff for noise-candles
    for candle_column in candle_columns:
        ohlc[f"previous_{candle_column}"] = ohlc[candle_column].shift(1)
        ohlc = _difference_from_first(dataframe=ohlc, boolean_column=f"is_{column}_greater_than_zero", value_column=f"previous_{candle_column}")
        ohlc.drop(labels=[f"previous_{candle_column}"], axis=1, inplace=True)

In [27]:
# global trend ratio up/down streak
for column in columns:
    ohlc[f"__previous"] = ohlc[column].shift(1)
    ohlc[f"is_{column}_greater_than_previous"] = ohlc.apply(
        lambda row: _is_greater(
            first_number=row[column], 
            second_number=row[f"__previous"]
        ), 
        axis=1
    )
    ohlc = _streak_by(dataframe=ohlc, boolean_column=f"is_{column}_greater_than_previous")
    ohlc.drop(labels=[f"__previous"], axis=1, inplace=True)

    # diff base
    for value_column in value_columns:
        ohlc = _difference_from_first(dataframe=ohlc, boolean_column=f"is_{column}_greater_than_previous", value_column=value_column)

    # diff for noise-candles
    for candle_column in candle_columns:
        ohlc[f"previous_{candle_column}"] = ohlc[candle_column].shift(1)
        ohlc = _difference_from_first(dataframe=ohlc, boolean_column=f"is_{column}_greater_than_previous", value_column=f"previous_{candle_column}")
        ohlc.drop(labels=[f"previous_{candle_column}"], axis=1, inplace=True)

In [28]:
len(list(ohlc.columns)) - 9

314